# Prompt Management System Demo

This notebook demonstrates how to use the prompt management system for compliance detection, including:

1. Loading prompts with the PromptManager
2. Creating and modifying prompts
3. Using advanced templates
4. Running A/B tests
5. Analyzing results with MLflow
6. Promoting winning prompts to production

In [ ]:
# Import necessary libraries
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml
import mlflow

# Add the parent directory to the path
project_root = Path.cwd().parents[2]
sys.path.append(str(project_root))

# Import prompt management tools
from llm_ops_pipeline.utils.prompt_management import PromptManager
from llm_ops_pipeline.utils.prompt_templates import PromptTemplate, PromptTemplateLibrary
from llm_ops_pipeline.utils.prompt_experimentation import PromptExperiment, load_few_shot_examples

## 1. Initialize the Prompt Management System

First, let's set up the prompt manager and template library.

In [ ]:
# Set up environment variables (or use from environment)
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"  # Replace with your MLflow server
# os.environ["LANGFUSE_API_KEY"] = "your-api-key"
# os.environ["LANGFUSE_SECRET_KEY"] = "your-secret-key"
# os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

# Initialize prompt manager
prompt_manager = PromptManager(
    langfuse_api_key=os.environ.get("LANGFUSE_API_KEY"),
    langfuse_secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    langfuse_host=os.environ.get("LANGFUSE_HOST"),
    environment="development"
)

# Initialize template library
prompts_dir = project_root / "prompts"
template_library = PromptTemplateLibrary(prompts_dir)

## 2. List Available Prompts

Let's look at what prompts are available in the system.

In [ ]:
# List all prompts
prompts = prompt_manager.list_prompts()

# Convert to DataFrame for better display
df_prompts = pd.DataFrame(prompts)
df_prompts

## 3. Working with Compliance Detection Prompts

Let's examine our compliance detection prompts.

In [ ]:
# Filter for compliance prompts
compliance_prompts = [p for p in prompts if "compliance" in p.get("id", "")]
compliance_prompts

In [ ]:
# Let's look at the few-shot prompt
try:
    few_shot_prompt = prompt_manager.get_prompt("compliance/few_shot")
    print(few_shot_prompt["content"])
except ValueError:
    print("Prompt not found. Let's create it from the template.")
    # Load the template
    few_shot_template = template_library.get_template("compliance/few_shot")
    
    # Load examples
    examples_path = project_root / "workflows/compliance_detection/prompts/few_shot_examples.json"
    few_shot_examples = load_few_shot_examples(str(examples_path))
    
    # Render with example data
    context = {
        "message": "This is a placeholder message.",
        "output_format": "json",
        "few_shot_examples": few_shot_examples[:3]  # Show just 3 examples
    }
    
    rendered = few_shot_template.render(context)
    print(rendered)

## 4. Creating a New Prompt Variant

Let's create a new prompt variant that combines step-by-step reasoning with few-shot examples.

In [ ]:
# Define the new template
hybrid_template = """
You are a compliance detection system for a financial institution. 
Analyze the provided message and determine if it contains any compliance breaches.

## Analysis Approach
To correctly classify this message, follow these steps:

1. Read the message carefully
2. Identify any potential compliance issues
3. Consider the following questions:
   - Does the message involve misleading clients or conflicts of interest?
   - Does the message suggest illegal activities like fraud or market manipulation?
   - Does the message violate regulatory requirements or compliance protocols?
   - Does the message improperly share confidential or sensitive information?
   - Does the message contain harassment or create a hostile work environment?
   - If none of the above apply, is the message professional and compliant?
4. Compare to similar examples below
5. Determine the primary category of violation
6. If multiple violations exist, classify based on the most severe one
7. If no violation exists, classify as COMPLIANT

## Classification Categories
- ETHICAL_BREACH: Conflicts of interest, misleading clients, dishonest behavior
- ILLEGAL_ACTIVITY: Fraud, money laundering, market manipulation
- REGULATORY_VIOLATION: Sharing material non-public information, violating compliance protocols
- CONFIDENTIAL_INFO: Improper sharing of client data, unreleased financial results, proprietary strategies
- HARASSMENT: Workplace harassment or inappropriate comments
- COMPLIANT: Professional and compliant communication

## Examples
{% for example in few_shot_examples %}
Message: "{{ example.message }}"
Category: {{ example.example_type }}
Explanation: {{ example.explanation }}

{% endfor %}

{% if output_format == "json" %}
Provide your classification as a JSON object with the following format:
```json
{
  "category": "CATEGORY_NAME",
  "confidence": 0.95,
  "reasoning": "Brief explanation of your classification"
}
```
{% else %}
Return only the category label with no additional text.
{% endif %}

## Message to Classify
"{{ message }}"
"""

# Create the new prompt in the manager
hybrid_prompt = prompt_manager.create_prompt(
    prompt_id="compliance/hybrid",
    content=hybrid_template,
    name="Hybrid Compliance Detection",
    description="Combined step-by-step reasoning with few-shot examples",
    tags=["compliance", "hybrid", "step_by_step", "few_shot"]
)

print(f"Created new prompt: {hybrid_prompt['metadata']['name']} (version {hybrid_prompt['metadata']['version']})")

## 5. Running Inference with Different Prompts

Let's test a message with different prompt variants to see how they compare.

In [ ]:
# Import modules for inference
from google.cloud import aiplatform
from vertexai.generative_models import GenerativeModel

# Load configuration
config_path = project_root / "workflows/compliance_detection/configs/compliance_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Initialize the model (assumes you have GCP credentials configured)
try:
    model_config = config.get("model", {})
    project_id = model_config.get("project_id")
    location = model_config.get("location", "us-central1")
    
    if project_id:
        aiplatform.init(project=project_id, location=location)
    
    model_name = model_config.get("name", "gemini-flash")
    model = GenerativeModel(model_name)
    print(f"✅ Initialized model: {model_name}")
except Exception as e:
    print(f"❌ Error initializing model: {e}")
    print("Will use a mock model for demonstration purposes")
    
    # Create a mock model class for demonstration
    class MockModel:
        def generate_content(self, prompt):
            class MockResponse:
                def __init__(self, text):
                    self.text = text
            
            # Return a simple simulated response
            return MockResponse(json.dumps({"category": "COMPLIANT", "confidence": 0.9, "reasoning": "Mock response"}))
    
    model = MockModel()

In [ ]:
# Define a function to classify messages
def classify_message(message, prompt_id, template_library=None):
    # Get the prompt from the manager
    try:
        prompt_data = prompt_manager.get_prompt(prompt_id)
        prompt_content = prompt_data.get("content", "")
    except ValueError:
        if not template_library:
            raise ValueError(f"Prompt '{prompt_id}' not found and no template library provided")
        
        # Try to get from template library
        template = template_library.get_template(prompt_id)
        prompt_content = template.template
    
    # Get examples if needed
    few_shot_examples = []
    if "few_shot" in prompt_id or "hybrid" in prompt_id:
        examples_path = project_root / "workflows/compliance_detection/prompts/few_shot_examples.json"
        try:
            few_shot_examples = load_few_shot_examples(str(examples_path))
        except Exception as e:
            print(f"Warning: Failed to load few-shot examples: {e}")
    
    # Render the template
    template = PromptTemplate(prompt_content, few_shot_examples=few_shot_examples)
    
    context = {
        "message": message,
        "output_format": "json",
        "few_shot_examples": few_shot_examples
    }
    
    rendered_prompt = template.render(context)
    
    # Generate completion
    start_time = pd.Timestamp.now()
    response = model.generate_content(rendered_prompt)
    latency = (pd.Timestamp.now() - start_time).total_seconds()
    
    completion = response.text.strip()
    
    # Parse the result
    try:
        json_str = completion
        if "```json" in completion:
            json_str = completion.split("```json")[1].split("```")[0].strip()
        elif "```" in completion:
            json_str = completion.split("```")[1].strip()
        
        result = json.loads(json_str)
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        result = {"category": "COMPLIANT", "error": str(e)}
    
    # Add metadata
    result["metadata"] = {
        "prompt_id": prompt_id,
        "latency": latency,
        "timestamp": pd.Timestamp.now().isoformat()
    }
    
    # Log prompt usage if MLflow is available
    try:
        prompt_manager.log_prompt_usage(
            prompt_id=prompt_id,
            inputs={"message": message},
            completion=completion,
            metadata={
                "latency": latency,
                "metrics": {
                    "latency": latency,
                    "tokens_used": len(rendered_prompt.split()) + len(completion.split())
                }
            }
        )
    except Exception as e:
        print(f"Warning: Failed to log prompt usage to MLflow: {e}")
    
    return result

In [ ]:
# Test different prompt variants on the same message
test_message = "I think we can structure these transactions into smaller amounts so they don't trigger the reporting requirements. That way our client won't have any issues with the authorities."

prompt_variants = [
    "compliance/basic",
    "compliance/detailed",
    "compliance/step_by_step",
    "compliance/few_shot",
    "compliance/hybrid"
]

results = {}
for variant in prompt_variants:
    print(f"Testing {variant}...")
    result = classify_message(test_message, variant, template_library)
    results[variant] = result
    print(f"  → {result['category']} (confidence: {result.get('confidence', 'N/A')})")
    print(f"  → Latency: {result['metadata']['latency']:.3f} seconds")
    print(f"  → Reasoning: {result.get('reasoning', 'None provided')}")
    print()

## 6. Setting up an A/B Test Experiment

Now let's set up an experiment to systematically compare our prompt variants.

In [ ]:
# Initialize an experiment
experiment = PromptExperiment(
    experiment_name="compliance-prompt-comparison",
    prompt_manager=prompt_manager,
    default_metrics=["accuracy", "latency", "confidence"]
)

# Add all prompt variants
for i, variant_id in enumerate(prompt_variants):
    variant_name = variant_id.split("/")[-1]
    experiment.add_variant(
        name=variant_name,
        prompt_id=variant_id,
        weight=1.0,  # Equal weights for fair comparison
        description=f"{variant_name.capitalize()} compliance detection prompt"
    )

print(f"Set up experiment with {len(prompt_variants)} variants")

In [ ]:
# Define the inference function for the experiment
def experiment_inference_fn(prompt, inputs, **kwargs):
    message = inputs["message"]
    true_label = inputs.get("true_label")
    
    # Get the variant and examples
    variant = kwargs.get("variant", "basic")
    
    # Get examples if needed
    few_shot_examples = []
    if "few_shot" in variant or "hybrid" in variant:
        examples_path = project_root / "workflows/compliance_detection/prompts/few_shot_examples.json"
        try:
            few_shot_examples = load_few_shot_examples(str(examples_path))
        except Exception as e:
            print(f"Warning: Failed to load few-shot examples: {e}")
    
    # Render the template
    template = PromptTemplate(prompt["content"], few_shot_examples=few_shot_examples)
    
    context = {
        "message": message,
        "output_format": "json",
        "few_shot_examples": few_shot_examples
    }
    
    rendered_prompt = template.render(context)
    
    # Generate completion
    response = model.generate_content(rendered_prompt)
    completion = response.text.strip()
    
    # Parse the result
    try:
        json_str = completion
        if "```json" in completion:
            json_str = completion.split("```json")[1].split("```")[0].strip()
        elif "```" in completion:
            json_str = completion.split("```")[1].strip()
        
        result = json.loads(json_str)
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        result = {"category": "COMPLIANT", "confidence": 0.0, "error": str(e)}
    
    # Add true label if available
    if true_label is not None:
        result["true_label"] = true_label
        result["correct"] = result["category"] == true_label
    
    return result

In [ ]:
# Load some test data
test_data_path = project_root / "data/compliance/test/data.csv"
if test_data_path.exists():
    test_df = pd.read_csv(test_data_path)
    print(f"Loaded {len(test_df)} test examples")
else:
    # Create some simple test examples
    print("Test data not found, creating synthetic examples")
    test_data = [
        {"text": "I'll make sure to tell those clients we're certain this investment will yield at least 15% returns, even though our models show only a 60% chance.", "label": "ETHICAL_BREACH"},
        {"text": "My contact at Company X just told me they're announcing a merger next week. We should buy a bunch of their stock today before the news goes public.", "label": "ILLEGAL_ACTIVITY"},
        {"text": "I've been skipping those know-your-customer checks for VIP clients. It saves us time and they always get annoyed by the questions anyway.", "label": "REGULATORY_VIOLATION"},
        {"text": "I've attached our complete client list with their portfolio values. Please use this to help identify new prospects for our service.", "label": "CONFIDENTIAL_INFO"},
        {"text": "That new analyst in the trading desk would be more valuable if she wore more flattering clothes. Someone should tell her how things work here.", "label": "HARASSMENT"},
        {"text": "I've completed the risk assessment for the Johnson portfolio and attached it for your review. Please let me know if you need any additional information.", "label": "COMPLIANT"}
    ]
    test_df = pd.DataFrame(test_data)

# Prepare inputs batch for experiment
inputs_batch = []
for _, row in test_df.iterrows():
    inputs_batch.append({
        "message": row["text"],
        "true_label": row["label"] if "label" in row else None
    })

# Take a small subset for the demo
inputs_batch = inputs_batch[:6]  # Limit to 6 examples for the notebook

In [ ]:
# Define evaluation function
def evaluation_fn(results):
    metrics = {
        "total": len(results),
        "error_count": sum(1 for r in results if "error" in r.get("results", {})),
        "latency": sum(r.get("metadata", {}).get("latency", 0) for r in results) / len(results)
    }
    
    # Calculate accuracy if true labels are available
    correct_count = sum(1 for r in results if r.get("results", {}).get("correct", False))
    metrics["accuracy"] = correct_count / len(results) if len(results) > 0 else 0
    metrics["error_rate"] = metrics["error_count"] / len(results) if len(results) > 0 else 0
    
    # Calculate average confidence
    confidence_values = [r.get("results", {}).get("confidence", 0) for r in results]
    metrics["confidence"] = sum(confidence_values) / len(confidence_values) if confidence_values else 0
    
    return metrics

In [ ]:
# Run the A/B test
with mlflow.start_run(experiment_id=experiment.experiment_id) as run:
    print(f"Running A/B test with MLflow run ID: {run.info.run_id}")
    
    results = experiment.run_ab_test(
        inputs_batch=inputs_batch,
        inference_fn=experiment_inference_fn,
        sampling_method="all",  # Test all variants on all inputs
        evaluation_fn=evaluation_fn
    )
    
    print("\nExperiment results:")
    for variant, variant_results in results.items():
        metrics = evaluation_fn(variant_results)
        print(f"\n{variant}:")
        print(f"  Accuracy: {metrics['accuracy']:.2f}")
        print(f"  Latency: {metrics['latency']:.3f} seconds")
        print(f"  Confidence: {metrics['confidence']:.2f}")

## 7. Analyzing Experiment Results from MLflow

Let's analyze the results of our experiment from MLflow.

In [ ]:
# Analyze the experiment results
analysis = experiment.analyze_experiment(min_samples=1)  # Use 1 for demo purposes

# Print the analysis
print("Experiment Analysis:\n")

# Print variant metrics
for variant, metrics in analysis.get("variants", {}).items():
    print(f"{variant}:")
    for metric_name, values in metrics.items():
        print(f"  {metric_name}: mean={values['mean']:.4f}, median={values['median']:.4f}, samples={values['samples']}")
    print()

# Print best variants
print("\nBest Variants:")
for metric, data in analysis.get("best_variants", {}).items():
    print(f"  {metric}: {data['variant']} ({data['value']:.4f})")

## 8. Promoting the Best Variant to Production

Based on our experiment results, let's promote the best variant to production.

In [ ]:
# Promote the best variant based on accuracy
best_metric = "accuracy"
if best_metric in analysis.get("best_variants", {}):
    best_variant = analysis["best_variants"][best_metric]["variant"]
    best_value = analysis["best_variants"][best_metric]["value"]
    
    print(f"Promoting {best_variant} to production (with {best_metric} = {best_value:.4f})")
    
    # Get the prompt ID
    prompt_id = f"compliance/{best_variant}"
    
    # Get existing prompt data
    prompt_data = prompt_manager.get_prompt(prompt_id)
    prompt_content = prompt_data.get("content", "")
    
    # Create or update production prompt
    production_prompt_id = "compliance/production"
    
    try:
        # Check if production prompt exists
        prompt_manager.get_prompt(production_prompt_id)
        
        # Update it
        updated_prompt = prompt_manager.update_prompt(
            prompt_id=production_prompt_id,
            content=prompt_content,
            description=f"Production compliance detection prompt (promoted from {best_variant})",
            tags=["compliance", "production", best_variant]
        )
        
        print(f"Updated production prompt (version {updated_prompt['metadata']['version']})")
        
    except ValueError:
        # Create new production prompt
        new_prompt = prompt_manager.create_prompt(
            prompt_id=production_prompt_id,
            content=prompt_content,
            name="Compliance Detection - Production",
            description=f"Production compliance detection prompt (promoted from {best_variant})",
            tags=["compliance", "production", best_variant]
        )
        
        print(f"Created new production prompt (version {new_prompt['metadata']['version']})")
else:
    print(f"No best variant found for {best_metric}")

## 9. Using the Production Prompt

Now let's use the production prompt for classification.

In [ ]:
# Test message for production prompt
production_message = "We need to move our positions before our competitors find out about this opportunity that was shared at the industry conference yesterday."

# Classify with production prompt
result = classify_message(production_message, "compliance/production", template_library)

print("Classification Result:")
print(f"Category: {result['category']}")
print(f"Confidence: {result.get('confidence', 'N/A')}")
print(f"Reasoning: {result.get('reasoning', 'None provided')}")
print(f"Latency: {result['metadata']['latency']:.3f} seconds")

## 10. Summary and Best Practices

In this notebook, we've demonstrated how to use the prompt management system for compliance detection, including:

1. **Prompt Management**: Using the PromptManager to store, retrieve, and version prompts
2. **Template System**: Creating advanced prompt templates with conditional logic and few-shot examples
3. **A/B Testing**: Running experiments to compare different prompt variants
4. **MLflow Integration**: Tracking performance metrics and managing experiments
5. **Production Deployment**: Promoting the best-performing prompt to production

Best practices for prompt engineering:

1. **Start with baseline**: Begin with a simple prompt and incrementally improve
2. **Use few-shot examples**: Include representative examples for better performance
3. **Structured reasoning**: Guide the model through a step-by-step reasoning process
4. **A/B test systematically**: Test multiple variants with the same inputs
5. **Track metrics**: Monitor performance over time to identify drift
6. **Version control**: Keep track of all prompt versions and their performance
7. **Provide clear instructions**: Be specific about the desired output format
8. **Fallback mechanisms**: Implement robust error handling and fallbacks